In [4]:
from openai import OpenAI
import os

openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

In [5]:
from rag_helper import RAGBase
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [6]:
instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [7]:
answer = assistant.rag("How do I run Ollama locally?")
print(answer)

To run Ollama on your own machine:

1. **Install Ollama**  
   - **macOS** – download the `.pkg` file from <https://ollama.com/download> and install it.  
   - **Windows** – download the `.msi` file from the same site and run the installer.  
   - **Linux** – run the installer script in a terminal:  

   ```bash
   curl -fsSL https://ollama.com/install.sh | sh
   ```

2. **Start a model locally**  
   Open a terminal (or a command‑prompt) and execute:

   ```bash
   ollama run llama3
   ```

   This command will:
   * Download the LLaMA 3 model (≈ 4 GB) if it isn’t already cached.  
   * Launch the model as a local server.  
   * Open a chat‑like interface where you can type prompts.

3. **Verify the local server**  
   In another terminal run:

   ```bash
   curl http://localhost:11434
   ```

   You should receive a JSON response such as:

   ```json
   {"models": [...]}
   ```

   This confirms the Ollama server is running on port 11434.

4. (Optional) **Use Ollama from Python**  
 

In [8]:
index.search("How do I run Ollama locally?")    

[{'id': '1d0b969028',
  'course': 'llm-zoomcamp',
  'section': 'Module 1: RAG',
  'question': 'Ollama: How to install Ollama?',
  'answer': 'First, install Ollama by visiting [https://ollama.com/download](https://ollama.com/download) and choosing your operating system:\n\n- **macOS**: Download the `.pkg` and install it.\n- **Windows**: Download the `.msi` and install it.\n- **Linux**: Run the following command in the terminal:\n\n  ```bash\n  curl -fsSL https://ollama.com/install.sh | sh\n  ```\n\nOnce installed, open a terminal and type:\n\n```bash\nollama run llama3\n```\n\nThis command will:\n\n- Download the LLaMA 3 model (~4GB).\n- Start the model locally.\n- Open a chat-like interface where you can type questions.\n\nTo test the Ollama local server, run the following command:\n\n```bash\ncurl http://localhost:11434\n```\n\nYou should receive a response similar to:\n\n```json\n{"models": [...]}  \n```\n\nThen, install the Python client with:\n\n```bash\npip install ollama\n```\n\n

In [9]:
answer = assistant.rag("How do I run Olama locally?")
print(answer)

Yes—you can run the course (including the Olama material) locally instead of using Codespaces. The only requirement is that you set up the same tooling that Codespaces provides:

1. **Install Python** (the version required by the course).  
2. **Install uv** (the fast Python package manager used in the notebooks).  
3. **Set up Jupyter** (JupyterLab or Notebook) so you can run the instructional notebooks.  
4. **Install Docker** (and any other container‑oriented tools the module depends on).  
5. **Install any additional libraries** the specific modules ask for (e.g., LangChain, Kestra plugins, etc.).

After you have everything installed:

- **Document your environment** (e.g., keep a `requirements.txt`, `pyproject.toml`, or `uv lockfile`).  
- **Make the setup reproducible** so you (and others) can recreate it later—using version‑pinned dependencies, Dockerfiles, or environment‑spec files.

Once your local environment mirrors the Codespaces setup, you can run all notebooks, scripts, a

In [10]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="openai/gpt-oss-120b",
    input=messages,
)

answer = response.output_text
print(answer)

Absolutely! You can join the course—just follow these simple steps:

1. **Visit the Course Page**  
   Go to the platform’s website or app where the course is listed. Look for the specific course title you’re interested in.

2. **Create or Sign In to Your Account**  
   - If you already have an account, log in with your credentials.  
   - If you don’t have an account yet, click “Sign Up” and fill in the required information (name, email, password, etc.). Most platforms also let you sign in with Google, Apple, or Facebook for convenience.

3. **Enroll in the Course**  
   - On the course’s landing page, you’ll usually see an “Enroll,” “Join,” or “Start Learning” button. Click it.  
   - Some courses are **free**, while others may require payment. If there’s a fee, you’ll be prompted to enter payment details (credit/debit card, PayPal, or another supported method).

4. **Confirm Your Enrollment**  
   After you’ve completed the enrollment step, you should receive a confirmation email (o

In [11]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [12]:
answers = search("How do I run Ollama locally?")
print(answers)

[{'id': '1d0b969028', 'course': 'llm-zoomcamp', 'section': 'Module 1: RAG', 'question': 'Ollama: How to install Ollama?', 'answer': 'First, install Ollama by visiting [https://ollama.com/download](https://ollama.com/download) and choosing your operating system:\n\n- **macOS**: Download the `.pkg` and install it.\n- **Windows**: Download the `.msi` and install it.\n- **Linux**: Run the following command in the terminal:\n\n  ```bash\n  curl -fsSL https://ollama.com/install.sh | sh\n  ```\n\nOnce installed, open a terminal and type:\n\n```bash\nollama run llama3\n```\n\nThis command will:\n\n- Download the LLaMA 3 model (~4GB).\n- Start the model locally.\n- Open a chat-like interface where you can type questions.\n\nTo test the Ollama local server, run the following command:\n\n```bash\ncurl http://localhost:11434\n```\n\nYou should receive a response similar to:\n\n```json\n{"models": [...]}  \n```\n\nThen, install the Python client with:\n\n```bash\npip install ollama\n```\n\nHere is 

In [13]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [14]:
print(f"original user message: {messages[0]['content']}")

original user message: I just discovered the course. Can I join it?


In [15]:
model = "openai/gpt-oss-120b"
# model = "llama-3.3-70b-versatile"
response = openai_client.responses.create(
    model=model,
    input=messages,
    tools=[search_tool],
)

response.output
print(len(response.output))
call0 = response.output[0]
call = response.output[1]

print(call0)
print(call)


2
ResponseReasoningItem(id='resp_01kwxtrfa2ftarzn7zaatpy9x1', summary=[], type='reasoning', content=[Content(text='We need to answer: user discovered the course, wants to join. Probably answer about enrollment, prerequisites, how to register. Use friendly tone. No need to call function.', type='reasoning_text')], encrypted_content=None, status='completed')
ResponseOutputMessage(id='msg_01kwxtrfa2ftbbctgsvv5q20rd', content=[ResponseOutputText(annotations=[], text='Welcome to the course! 🎉\n\nYes—you can definitely join. Here’s what you need to know to get started:\n\n### 1. Check the enrollment criteria\n- **Prerequisites:** Most of our courses are open‑entry, but a few advanced modules may require prior knowledge (e.g., basic programming or introductory statistics). The course description will list any required background.\n- **Eligibility:** The program is open to anyone over 18 (or with parental consent if you’re younger). If you’re a current student, employee, or just an interested 

In [16]:
print(call.name)

AttributeError: 'ResponseOutputMessage' object has no attribute 'name'

In [17]:
import json
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

AttributeError: 'ResponseOutputMessage' object has no attribute 'arguments'

In [ ]:
search_results = search(args["query"])
print(search_results)

In [ ]:
messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

In [ ]:
print(messages)

In [18]:
response = openai_client.responses.create(
    model=model,
    input=messages,
    tools=[search_tool],
)

response.output_text

'Absolutely! Our courses are open to anyone who’s interested, and getting started is straightforward. Here’s a quick rundown of what you need to do to join:\n\n### 1. Check the Prerequisites (if any)\n- Most of our courses are **open‑access**, meaning no prior experience is required.\n- A few advanced tracks may list specific prerequisites (e.g., basic programming knowledge, familiarity with statistics, etc.). You can find those details on the course description page.\n\n### 2. Create an Account\n- Go to the **Sign‑Up** page on our platform.\n- Fill in your name, email address, and create a password.\n- You’ll receive a verification email—click the link inside to activate your account.\n\n### 3. Enroll in the Course\n- Once you’re logged in, browse the **Course Catalog** and locate the course you discovered.\n- Click **Enroll** (or **Add to Cart** if the course has a fee).\n- If it’s a paid course, you’ll be prompted to enter payment details. Many of our courses are **free**, and for p

In [19]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(151, 774)

In [ ]:
def calculate_cost(input_tokens, output_tokens):
    """
    Calculate the cost of using the GPT-OSS-120B model.
    Args:
    - input_tokens (int): The number of input tokens.
    - output_tokens (int): The number of output tokens.
    Returns:
    - cost (float): The total cost in dollars.
    """
    # Pricing in dollars per million tokens
    input_cost_per_million = 0.1500
    output_cost_per_million = 0.6000
    # Calculate cost for input and output tokens
    input_cost = (input_tokens / 1e6) * input_cost_per_million
    output_cost = (output_tokens / 1e6) * output_cost_per_million
    # Total cost
    total_cost = input_cost + output_cost
    return total_cost

In [ ]:
cost = calculate_cost(usage.input_tokens, usage.output_tokens)
print(f"Total cost for this interaction: ${cost:.6f}")

# Agentic loop

In [ ]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [ ]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [ ]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.responses.create(
    model=model,
    input=messages,
    tools=[search_tool],
)

messages.extend(response.output)
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

function_call: search {"query":"Can I join the course? enrollment eligibility"}


Full agent loop

In [ ]:
def agent_loop(instructions, question, model=model) -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [ ]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {"query":"Olama locally run"}
iteration #2...
function_call: search {"query":"Can I run the course locally instead of Codespaces?"}
iteration #3...
function_call: search {"query":"Olama"}
iteration #4...
function_call: search {"query":"Olama run locally"}
iteration #5...
function_call: search {"query":"Ollama run locally"}
iteration #6...
ASSISTANT:
Below is a short, step‑by‑step guide to get **Ollama** up and running on your own machine and to use it from the command line or from Python code (as we do in the Zoomcamp notebooks).

---

## 1️⃣ Install Ollama

| OS | How to install |
|----|----------------|
| **macOS** | Download the `*.pkg` installer from <https://ollama.com/download> and double‑click to install. |
| **Windows** | Download the `*.msi` installer from the same page and run it. |
| **Linux** | Run the one‑liner in a terminal: <br>`curl -fsSL https://ollama.com/install.sh \| sh` |

*(If you prefer Docker, you can also run `docker run -d

'Below is a short, step‑by‑step guide to get **Ollama** up and running on your own machine and to use it from the command line or from Python code (as we do in the Zoomcamp notebooks).\n\n---\n\n## 1️⃣ Install Ollama\n\n| OS | How to install |\n|----|----------------|\n| **macOS** | Download the `*.pkg` installer from <https://ollama.com/download> and double‑click to install. |\n| **Windows** | Download the `*.msi` installer from the same page and run it. |\n| **Linux** | Run the one‑liner in a terminal: <br>`curl -fsSL https://ollama.com/install.sh \\| sh` |\n\n*(If you prefer Docker, you can also run `docker run -d -p 11434:11434 ollama/ollama` – the same ports and API are exposed.)*\n\nAfter installation the `ollama` binary should be on your `$PATH`. Test it:\n\n```bash\nollama --version\n```\n\nYou should see something like `ollama 0.1.40` (or newer).\n\n---\n\n## 2️⃣ Start the Ollama server (optional but recommended)\n\nThe server is automatically started the first time you run a 

In [ ]:
agent_loop(instructions, "I just discovered the course. Can I still join it?")

iteration #1...
function_call: search {"query":"Can I still join the course after it has started"}
iteration #2...
ASSISTANT:
**Answer:**  
Yes—you can still join the course even if you discovered it after it started. The only condition for receiving a certificate is that you submit your capstone project while the submission form is still open. As long as the form hasn’t been closed, you can enroll, work through the material, and turn in your project to qualify for the certificate.

If you have any other questions (e.g., about homework deadlines, peer‑review requirements, or how the leaderboard works), just let me know!


'**Answer:**  \nYes—you can still join the course even if you discovered it after it started. The only condition for receiving a certificate is that you submit your capstone project while the submission form is still open. As long as the form hasn’t been closed, you can enroll, work through the material, and turn in your project to qualify for the certificate.\n\nIf you have any other questions (e.g., about homework deadlines, peer‑review requirements, or how the leaderboard works), just let me know!'

In [ ]:
!uv add toyaikit

Resolved 127 packages in 4.25s                                       
⠙ Preparing packages... (0/7)                                                   ⠋ Preparing packages... (0/0)                                                   
⠙ Preparing packages... (0/7)-------------------     0 B/74.86 KiB           
⠹ Preparing packages... (0/7)------------------- 14.81 KiB/74.86 KiB         
⠹ Preparing packages... (0/7)------------------- 14.81 KiB/74.86 KiB         
⠹ Preparing packages... (0/7)------------------- 14.81 KiB/74.86 KiB         
httpx2               ------------------------------ 14.81 KiB/74.86 KiB
⠹ Preparing packages... (0/7)-------------------     0 B/934.47 KiB          
httpx2               ------------------------------ 14.81 KiB/74.86 KiB
⠹ Preparing packages... (0/7)------------------- 14.83 KiB/934.47 KiB        
httpx2               ------------------------------ 14.81 KiB/74.86 KiB
⠹ Preparing packages... (0/7)------------------- 14.83 KiB/934.47 KiB        
httpx2 

Restricting off-topic questions

In [ ]:
agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"Queen's Gambit opening definition"}
iteration #2...
function_call: search {"query":"queen gambit chess opening"}
iteration #3...
ASSISTANT:
**The Queen’s Gambit (chess opening)**  

| Move order | Name | What White is trying to do |
|------------|------|----------------------------|
| **1. d4 d5 2. c4** | Queen’s Gambit | Offer the pawn on c4 to distract Black’s d‑pawn, so White can gain a strong foothold in the centre with e4 or a lasting spatial advantage. |

---

### 1. Why it’s called a “gambit”

* **Gambit** = a deliberate sacrifice (or offer) of material early in the game.  
* In the Queen’s Gambit White *offers* the pawn on c4.  
* Most strong players do **not** actually lose the pawn – if Black accepts (2…dxc4) White can often recover it with a tempo‑winning move (e.g., 3.Nf3 c6 4.e3 b5 5.Bxc4) or simply build a huge centre with e4 later on.  

---

### 2. Main branches

| Branch | Typical continuation | Key ideas |
|--------|----

'**The Queen’s\u202fGambit (chess opening)**  \n\n| Move order | Name | What White is trying to do |\n|------------|------|----------------------------|\n| **1.\u202fd4\u202fd5 2.\u202fc4** | Queen’s Gambit | Offer the pawn on c4 to distract Black’s d‑pawn, so White can gain a strong foothold in the centre with e4 or a lasting spatial advantage. |\n\n---\n\n### 1. Why it’s called a “gambit”\n\n* **Gambit** = a deliberate sacrifice (or offer) of material early in the game.  \n* In the Queen’s Gambit White *offers* the pawn on c4.  \n* Most strong players do **not** actually lose the pawn – if Black accepts (2…dxc4) White can often recover it with a tempo‑winning move (e.g.,\u202f3.Nf3\u202fc6\u202f4.e3\u202fb5\u202f5.Bxc4) or simply build a huge centre with e4 later on.  \n\n---\n\n### 2. Main branches\n\n| Branch | Typical continuation | Key ideas |\n|--------|---------------------|-----------|\n| **Queen’s Gambit Accepted (QGA)** | 1.d4 d5 2.c4 **dxc4** | Black grabs the pawn. White a

In [ ]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit"}
iteration #2...
ASSISTANT:
I’m sorry, but I don’t have information on that topic in the course FAQ. Is there something else related to the course that I can help you with?


'I’m sorry, but I don’t have information on that topic in the course FAQ. Is there something else related to the course that I can help you with?'

## ToyAIkit

In [20]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [23]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [22]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [24]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [25]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [26]:
llm_client = OpenAIClient(
    model=model,
    client=openai_client
)

In [27]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=llm_client
)

In [28]:
result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)

-> Response received


-> Response received


-> Response received


-> Response received


/workspaces/LLM-zoomcamp-01-agentic-rag/.venv/lib/python3.12/site-packages/toyaikit/chat/runners.py:283: UnknownModelWarning: No pricing data for model 'openai/gpt-oss-120b'. Register it with PricingConfig.register_model(...) to get cost calculations.
  cost_info = self.pricing_config.calculate_cost(


In [30]:
print(result.cost)

None


In [31]:
result.all_messages

[EasyInputMessage(content="You're a course teaching assistant.\nAnswer the QUESTION based on the CONTEXT from the FAQ database.\nUse only the facts from the CONTEXT when answering the QUESTION.", role='developer', phase=None, type=None),
 EasyInputMessage(content='How do I run Olama locally?', role='user', phase=None, type=None),
 ResponseReasoningItem(id='resp_01kwxtxqrqe129zkg5pk2ta5fx', summary=[], type='reasoning', content=[Content(text='The user asks: "How do I run Olama locally?" We need to answer based on the FAQ database. We need to search for relevant entries. Use the search function.', type='reasoning_text')], encrypted_content=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"Olama run locally"}', call_id='fc_8001d848-099a-4a63-8092-8eb2d3e476d8', name='search', type='function_call', id='fc_8001d848-099a-4a63-8092-8eb2d3e476d8', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'fc_8001d848-099a-4a63-8092-8eb2d3e476d8

In [32]:
result2 = runner.loop(
    prompt="How do I run a different model?",
    previous_messages=result.all_messages,
    callback=callback,
)

-> Response received


/workspaces/LLM-zoomcamp-01-agentic-rag/.venv/lib/python3.12/site-packages/toyaikit/chat/runners.py:283: UnknownModelWarning: No pricing data for model 'openai/gpt-oss-120b'. Register it with PricingConfig.register_model(...) to get cost calculations.
  cost_info = self.pricing_config.calculate_cost(


In [33]:
runner.run()

-> Response received


/workspaces/LLM-zoomcamp-01-agentic-rag/.venv/lib/python3.12/site-packages/toyaikit/chat/runners.py:283: UnknownModelWarning: No pricing data for model 'openai/gpt-oss-120b'. Register it with PricingConfig.register_model(...) to get cost calculations.
  cost_info = self.pricing_config.calculate_cost(


-> Response received


-> Response received


-> Response received


...

-> Response received


-> Response received


Chat ended.


/workspaces/LLM-zoomcamp-01-agentic-rag/.venv/lib/python3.12/site-packages/toyaikit/chat/runners.py:184: UnknownModelWarning: No pricing data for model 'openai/gpt-oss-120b'. Register it with PricingConfig.register_model(...) to get cost calculations.
  combined_cost = self.pricing_config.calculate_cost(


LoopResult(new_messages=[EasyInputMessage(content="You're a course teaching assistant.\nAnswer the QUESTION based on the CONTEXT from the FAQ database.\nUse only the facts from the CONTEXT when answering the QUESTION.", role='developer', phase=None, type=None), EasyInputMessage(content='hi', role='user', phase=None, type=None), ResponseReasoningItem(id='resp_01kwxvdfmyfkj981evjnmn20qm', summary=[], type='reasoning', content=[Content(text="We need to answer the user's greeting. Probably respond as a teaching assistant, greet back and ask how can help. Use no external info.", type='reasoning_text')], encrypted_content=None, status='completed'), ResponseOutputMessage(id='msg_01kwxvdfmyfkjr2e14gxzg5ft1', content=[ResponseOutputText(annotations=[], text='Hello! How can I help you today?', type='output_text', logprobs=None)], role='assistant', status='completed', type='message', phase=None), EasyInputMessage(content='olama?', role='user', phase=None, type=None), ResponseReasoningItem(id='res